# 모델의 요청을 실행하고 결과를 돌려주는 루프

한 답변에 도구 요청이 두 개 들어왔을 때, 조회 결과의 도착 순서가 달라져도 원래 요청과 짝을 맞춘다.

### 노트북 실행 준비

Python 3.12 커널에서 위쪽 셀부터 실행하세요. 이 예제는 파이썬 표준 라이브러리로 실행할 수 있습니다.

## 모델 출력과 프로그램의 책임

모델이 함수를 골랐을 때 생기는 것은 함수 이름과 인자가 적힌 요청이다. 실제 실행은 내 프로그램이 허용된 함수와 인자를 확인한 뒤 맡는다.

도서관 운영 시간과 대출 기간을 함께 물으면 서로 다른 두 조회가 필요하다. 먼저 끝난 조회를 첫 요청의 답으로 붙이면 문장이 자연스러워도 근거가 바뀐다.

요청 ID는 한 번의 호출을 식별하고 함수 이름은 할 일을 설명한다. 같은 이름으로 서로 다른 인자를 보낸 요청이 두 건 올 수 있으므로 이름 하나로 결과를 찾으면 부족하다.

## 늦게 도착한 응답 연결하기

공급자 응답에서 꺼낸 가상 호출 기록이다. 여기서는 SDK 통신 이후에 우리 코드가 맡는 결합 단계만 실행한다.

In [1]:
calls = [("c1", "운영 시간"), ("c2", "대출 기간")]
received = [("c2", "14일"), ("c1", "18시까지")]
def pair_results(calls, received):
    replies = dict(received)
    return [(name, replies.get(call_id)) for call_id, name in calls]

print(pair_results(calls, received))

[('운영 시간', '18시까지'), ('대출 기간', '14일')]


응답 순서가 반대여도 c1에 운영 시간을 연결한다. 없는 호출의 반환값까지 있는 것처럼 모델에 넘기면 재시도 판단이 어려워진다.

### 돌아오지 않은 요청 하나 찾기

c2만 도착한 상태다. 두 도구를 전부 다시 실행하지 않고 대기 중인 호출 ID 한 개를 찾는다.

In [2]:
partial = received[:1]

In [3]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>arrived = {call_id for call_id, value in partial}
print([call_id for call_id, name in calls if call_id not in arrived])</code></pre><pre>[&#x27;c1&#x27;]
</pre><p>c1이 남는다. 읽기 전용 조회를 재시도할 때도 최대 시도를 정해 두고, 저장을 수행하는 도구라면 실제 저장 여부를 먼저 조사한다.</p></details>

## 같은 호출 ID로 다른 결과가 오면?

딕셔너리를 한 번에 만들면 뒤의 값이 앞의 값을 덮는다. 반환 기록을 결합하기 전에 요청 목록과 응답의 식별자를 검사한다.

In [4]:
def reconcile(calls, results):
    requested = {key for key, name in calls}
    matched = {}
    for key, value in results:
        if key not in requested:
            raise ValueError("알 수 없는 호출 " + key)
        if key in matched and value != matched[key]:
            raise ValueError("서로 다른 반환값 " + key)
        matched[key] = value
    return matched

print(reconcile(calls, received + [received[0]]))

{'c2': '14일', 'c1': '18시까지'}


내용이 같은 재전송은 한 번만 반영한다. 반환값이 다른 두 결과를 마지막 도착 순서로 결정하면 담당자가 근거를 고를 수 없다.

### 충돌한 도구 응답을 모델에 넘기지 않기

c2의 결과가 처음에는 14일, 재전송 때 7일이다. 어느 호출을 보류해야 하는지 예외에서 읽고, 새 호출 ID와 같은 결과인 경우에도 검사를 유지한다.

In [5]:
bad = received + [("c2","7일")]

In [6]:
# 이 셀에 풀이를 작성하세요.

<details class="notebook-answer"><summary>풀이와 실행 결과</summary><pre><code>try:
    reconcile(calls, bad)
except ValueError as error:
    print(error)</code></pre><pre>서로 다른 반환값 c2
</pre><p>c2를 검토 대기 상태로 보낼 수 있다. 호출 이름이 아니라 요청 ID를 기록했기 때문에 c1의 정상 결과는 추적할 수 있다.</p></details>

## 다른 입력에 적용하기

SDK 문의 처리 노트북에서 ID가 없는 문의와 조회 결과가 없는 문의를 보낸다. 첫 문의는 다시 질문하고, 두 번째는 도구가 반환한 빈 결과를 설명하도록 연결한다. LCEL 예제에서도 어느 단계가 모델을 부르고 어느 단계가 단순 변환인지 표시한다.

최종 문장 옆에 실제 호출 횟수, 인자, 반환값, 종료 이유를 기록한다. 도구 하나가 예외를 냈을 때 이미 성공한 조회까지 다시 실행하는지 확인한다.

## 실제 응답에서 두 조회 결과 연결하기

운영 시간과 대출 기간을 한 번에 묻는다. 모델이 고른 도구와 ID는 실제 응답에서 꺼낸다.

수업 커널에 langchain-openai와 langchain을 설치하고 OPENAI_API_KEY를 환경 변수에 설정한다. 호출에는 API 요금이 발생한다.

In [ ]:
import os
from langchain_openai import ChatOpenAI
model = ChatOpenAI(model=os.getenv("OPENAI_MODEL") or "gpt-4o-mini", timeout=45, max_retries=1)
from langchain.agents import create_agent
def hours() -> str:
    """주말의 도서관 운영 시간을 조회한다."""
    return "10:00~16:00"
def loan() -> str:
    """도서 대출 기간을 조회한다."""
    return "14일"
agent = create_agent(model, tools=[hours,loan], system_prompt="운영 시간과 대출 기간은 각각 해당 도구로 확인하세요.")
log = agent.invoke({"messages":[("user","주말 시간과 대출 기간을 알려 주세요.")]})["messages"]
for m in log:
    print(m.type, getattr(m,"tool_calls",None) or m.content)

두 질문 중 하나만 남겨 실제 선택된 도구와 답변이 어떻게 달라지는지 확인한다.